In [ ]:
import os
import glob
import win32com.client as win32

# paths
downloads = r"path1"
arquivos_iqy = glob.glob(os.path.join(downloads, "*.iqy"))

if not arquivos_iqy:
    raise ValueError("Nenhum arquivo .iqy encontrado na pasta Downloads.")

# Catch the most recent file
ultimo_arquivo = max(arquivos_iqy, key=os.path.getctime)
print(f"Processando arquivo: {ultimo_arquivo}")

# Name of output
caminho_saida_xlsx = os.path.splitext(ultimo_arquivo)[0] + ".xlsx"

# Start Excel for automation
excel_app = win32.Dispatch("Excel.Application")
excel_app.Visible = False
excel_app.DisplayAlerts = False  # Evita pop-ups de segurança do .iqy

try:
    # Open the iqy and refresh
    wb = excel_app.Workbooks.Open(ultimo_arquivo)
    
    print("Atualizando dados (RefreshAll)...")
    wb.RefreshAll()
    
    # Wait the finish in second screen
    excel_app.CalculateUntilAsyncQueriesDone()
    
    # save excel default (.xlsx = FileFormat 51)
    # If the file already exists, it will be silently overwritten because of the DisplayAlerts
    wb.SaveAs(caminho_saida_xlsx, FileFormat=51)
    print(f"Sucesso! Arquivo convertido para: {caminho_saida_xlsx}")

except Exception as e:
    print(f"Erro durante o processamento: {e}")

finally:
    # close with security
    if 'wb' in locals():
        wb.Close(SaveChanges=False)
    excel_app.Quit()

In [ ]:
import pandas as pd

# Paths (Using the correct variable from the previous block)
# the name is caminho_saida_xlsx
arquivo_novo = caminho_saida_xlsx 
arquivo_base = r"path2"

print(f"Lendo dados de: {arquivo_novo}")

# read files
df_novo = pd.read_excel(arquivo_novo)
df_base = pd.read_excel(arquivo_base)

# Standardazating columns
df_novo.columns = df_novo.columns.str.strip()
df_base.columns = df_base.columns.str.strip()

# Align columns
# catching only database columns that you can
colunas_comuns = [col for col in df_novo.columns if col in df_base.columns]
df_novo_filtrado = df_novo[colunas_comuns].copy() # .copy() evita o SettingWithCopyWarning

# Ensure that the missing columns are filled in..
for col in df_base.columns:
    if col not in df_novo_filtrado.columns:
        df_novo_filtrado[col] = None

# reorder and concatenate and save overwrittening the db
df_novo_filtrado = df_novo_filtrado[df_base.columns]
df_final = pd.concat([df_base, df_novo_filtrado], ignore_index=True)
df_final.to_excel(arquivo_base, index=False)

print(f"Data successfully inserted into the history! total number of rows now {len(df_final)}")